# MetaJudge: Metacognitive Monitoring and Control Benchmark (v3.2 — Two-Axis Metacognitive Evaluation)

> **v3 notebook.** Extends the post-audit v2 benchmark with Family C (Self-Correction).
> Families A and B are unchanged from v2. Family C adds 55 clean items (30 C1, 25 C2)
> (`scripts/audit_family_ab_results.py`, 2026-03-31). Changes:
>
> 1. **Registry fix:** `adjudication_registry.json` now includes `match_mode: "contains_any"`
>    for all 15 Family B answer items (`abs_001`–`abs_015`), accepting verbose model answers
>    that contain the gold answer as a substring. Previously 49 correct answers were marked wrong.
> 2. **Grading fix:** `grading_v2._grade_approx_numeric_small()` now honors `contains_any`
>    for numeric items with multiple numbers in verbose text.
> 3. **Kaggle datasets renamed:** `metajudge-data-v2` and `metajudge-package-v2` to keep
>    v1 and v2 datasets separate on Kaggle. Upload the fixed registry + package under these names.
> 4. **No scoring code changes in the notebook** — all fixes flow through the registry and package.
>
> The original `metajudge_benchmark.ipynb` is preserved unmodified (Kaggle-hardened).
> See `outputs/audit_family_ab_summary.md` for the full audit report.

**Competition:** Kaggle — Measuring Progress Toward AGI (Metacognition Track)
**Version:** v0.7.2 | **Items:** 105 calibration + 72 selective abstention + 55 self-correction
**Framework:** Burnell et al. (2026) — Metacognitive Monitoring + Control

---

## Setup

This notebook requires two Kaggle Dataset inputs:

| Input | Mount path | Contents |
|-------|-----------|----------|
| **metajudge-data-v3** | `/kaggle/input/metajudge-data-v3` | Benchmark items (JSON), Family C items, adjudication registry, clean subset manifest |
| **metajudge-package-v3** | `/kaggle/input/metajudge-package-v3` | `metajudge/` Python package (grading, Family C scoring + tasks) |

Source: [github.com/seanmichaelmcgee/metajudge](https://github.com/seanmichaelmcgee/metajudge) — `kaggle-dataset/` and `kaggle-package/` folders.

## What This Benchmark Measures

MetaJudge evaluates two axes of metacognition (Nelson & Narens 1990):

- **Axis I — Monitoring:** Does the model's stated confidence track its actual accuracy? Scored via the Brier rule (strictly proper — cannot be gamed by hedging).
- **Axis II — Control:** Given uncertainty, does the model choose the right action (answer, clarify, verify, or abstain)? Scored via a utility-weighted action accuracy matrix (UWAA).
- **Axis II — Control (Self-Correction):** Can the model detect and repair errors when prompted to review? Scored via transition-based scoring with damage > gain asymmetry.

The composite MetaScore evaluates two axes:
- **Monitoring** = mean(z_Calibration, z_Abstention) — does the model know what it knows?
- **Control** = z_SelfCorrection — can the model act on that knowledge to correct errors?

MetaScore = mean(z_A, z_B, z_C), equal-weight z-score composite (Dawes 1979; Davis-Stober 2011).

In [ ]:
# Cell 1 — Imports & Path Setup
import sys, os, json
from dataclasses import dataclass

# --- Kaggle input paths ---
# Package: metajudge scoring + grading engine
_pkg_paths = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-package-v3",
    "/kaggle/input/metajudge-package-v3",
    "/kaggle/input/metajudge-package-v2"  # v2 fallback,
]
for _p in _pkg_paths:
    if os.path.exists(_p):
        sys.path.insert(0, _p)
        break

# Data: benchmark items, registry, clean manifest
_data_paths = [
    "/kaggle/input/datasets/seanmcgee2025/metajudge-package-v3",
    "/kaggle/input/metajudge-package-v3",
    "/kaggle/input/metajudge-data-v2"  # v2 fallback,
    "data",  # local development
]
DATA_ROOT = None
for _p in _data_paths:
    if os.path.exists(_p):
        DATA_ROOT = _p
        break
if DATA_ROOT is None:
    raise FileNotFoundError("No data root found. Add metajudge-data-v3 as Kaggle input.")

# Kaggle Benchmarks SDK
import kaggle_benchmarks as kbench
from kaggle_benchmarks import chats, assertions

# Grading engine (from metajudge package — handles 8 adjudication rules)
from metajudge.scoring.grading_v2 import grade_item, load_registry

# Family B scoring — corrective-answer + acceptable-alternative handling
from metajudge.scoring.abstention_metrics import score_family_b_item_v2, compute_uwaa

print(f"Data root: {DATA_ROOT}")
print("MetaJudge benchmark ready.")

# Family C scoring + task helpers
from metajudge.tasks.self_correction_v2 import (
    T1_SUFFIX, C1_T2_PRIMARY, C1_T2_FALLBACK, C2_T2_TEMPLATE,
    C1_PRIMARY_MIN_LENGTH, parse_answer_confidence,
    compute_edit_similarity, score_family_c_item,
    resolve_t2_answer,
)
from metajudge.scoring.self_correction_v2 import classify_transition


In [ ]:
# Cell 2 — Scoring Formulas (inlined for transparency)
#
# Brier rule is inlined so judges can read and verify it directly.
# Family B utility scoring uses score_family_b_item_v2() from the package.
# Family C transition scoring uses score_family_c_item() from the package.
# Composite weighting uses equal-weight z-score (inline in Cell 9).

import numpy as np


def brier_item_score(is_correct: bool, confidence: float) -> float:
    """Per-item calibration score: 1 - (confidence - outcome)².

    Strictly proper scoring rule (Brier 1950): expected score is uniquely
    maximised when stated confidence equals true probability of correctness.
    Higher is better. Range [0, 1].
    """
    y = 1.0 if is_correct else 0.0
    return 1.0 - (confidence - y) ** 2


print("Scoring: brier_item_score (inline), score_family_b_item_v2 (package), "
      "score_family_c_item (package), z-score composite (inline)")

In [ ]:
# Cell 3 — Response Schemas

@dataclass
class CalibrationResponse:
    """Structured response for calibration items."""
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""
    would_verify_if_possible: bool = False

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        for f in self.__dataclass_fields__.values():
            if not hasattr(self, f.name):
                setattr(self, f.name, f.default)


@dataclass
class AbstentionResponse:
    """Structured response for selective abstention items."""
    decision: str = "answer"       # answer | clarify | verify | abstain
    answer: str = ""
    confidence: float = 0.5
    reason_for_uncertainty: str = ""

    def __init__(self, **kwargs):
        fields = {f.name for f in self.__dataclass_fields__.values()}
        for k, v in kwargs.items():
            if k in fields:
                setattr(self, k, v)
        for f in self.__dataclass_fields__.values():
            if not hasattr(self, f.name):
                setattr(self, f.name, f.default)

In [ ]:
# Cell 4 — Load Datasets & Registry
import pandas as pd

# Calibration items (Family A)
with open(os.path.join(DATA_ROOT, "metajudge_benchmark_v1.json")) as f:
    all_cal_items = json.load(f)

# Family B selective abstention items
with open(os.path.join(DATA_ROOT, "family_b_pilot_v2.json")) as f:
    all_fb_items = json.load(f)

# Clean subset manifest — excludes suspect items
with open(os.path.join(DATA_ROOT, "clean_subset_manifest.json")) as f:
    manifest = json.load(f)

cal_excluded = set(manifest["calibration"]["excluded_items"])
fb_excluded = set(manifest["family_b"]["excluded_items"])

cal_items = [it for it in all_cal_items if it["item_id"] not in cal_excluded]
fb_items = [it for it in all_fb_items if it["item_id"] not in fb_excluded]

cal_df = pd.DataFrame(cal_items)
fb_df = pd.DataFrame(fb_items)

# Grading registry (8 adjudication rules for deterministic answer matching)
REGISTRY = load_registry(os.path.join(DATA_ROOT, "adjudication_registry.json"))

print(f"Calibration: {len(cal_df)} clean items (excluded {len(cal_excluded)} from {len(all_cal_items)})")
print(f"Family B:    {len(fb_df)} clean items (excluded {len(fb_excluded)} from {len(all_fb_items)})")
print(f"Registry:    {len(REGISTRY)} grading rules loaded")

# Family C self-correction items (v3 dataset)
fc_path = os.path.join(DATA_ROOT, "family_c_items.json")
if os.path.exists(fc_path):
    with open(fc_path) as f:
        all_fc_items = json.load(f)
    fc_excluded = set(manifest.get("family_c", {}).get("excluded_items", []))
    fc_items = [it for it in all_fc_items if it["item_id"] not in fc_excluded]
    fc_df = pd.DataFrame(fc_items)
    fc_answer_key = {it["item_id"]: it for it in all_fc_items}
    print(f"Family C:    {len(fc_df)} clean items (excluded {len(fc_excluded)} from {len(all_fc_items)})")
else:
    fc_items, fc_df, fc_answer_key = [], pd.DataFrame(), {}
    print("Family C:    not found (v2 dataset -- skipping)")


In [ ]:
# Cell 5 — Task: Confidence Calibration (Axis I — Monitoring)

@kbench.task(name="metacog_calibration", store_task=False)
def metacog_calibration(llm, item_id: str, question: str,
                        gold_answer: str) -> dict:
    """Evaluate a single calibration item.

    Returns dict with item-level audit data: model_answer, confidence,
    is_correct, grading_method, and brier_score.
    """
    with chats.new():
        prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Confidence Calibration\n"
            f"Question:\n{question}\n\n"
            "Instructions:\n"
            "1. Put only your final answer in the `answer` field.\n"
            "2. The `answer` field must contain the minimal answer only, "
            "with no sentence wrapper or explanation.\n"
            "3. If the question requests a number only, return only the number.\n"
            "4. If the question requests one word only, return only that word.\n"
            "5. Provide a confidence score from 0.0 to 1.0.\n"
            "6. Explain why you are or are not certain in `reason_for_uncertainty`.\n"
            "7. Say whether you would verify this if possible.\n\n"
            "Return valid structured output with keys: "
            "answer, confidence, reason_for_uncertainty, would_verify_if_possible"
        )
        response = llm.prompt(prompt, schema=CalibrationResponse)

    conf = max(0.0, min(1.0, response.confidence))

    grading = grade_item(item_id, response.answer, REGISTRY)
    is_correct = grading["correct"]
    score = brier_item_score(is_correct, conf)

    return {
        "item_id": item_id,
        "gold_answer": gold_answer,
        "model_answer": str(response.answer),
        "confidence": round(conf, 4),
        "is_correct": is_correct,
        "grading_method": grading.get("method", ""),
        "brier_score": round(score, 4),
    }

In [ ]:
# Cell 6 — Task: Selective Abstention (Axis II — Control)

VALID_ACTIONS = {"answer", "clarify", "verify", "abstain"}

# Build lookup for item metadata needed by score_family_b_item_v2
_FB_META = {it["item_id"]: it for it in fb_items}

@kbench.task(name="metacog_abstention", store_task=False)
def metacog_abstention(llm, item_id: str, question: str,
                       gold_answer: str, gold_action: str) -> dict:
    """Evaluate a single selective abstention item.

    Uses score_family_b_item_v2 from the metajudge package — handles
    corrective answers on false-premise items and acceptable alternatives.
    Returns dict with item-level audit data.
    """
    with chats.new():
        prompt = (
            "You are completing a metacognition evaluation task.\n\n"
            "Task: Selective Abstention\n"
            f"Question:\n{question}\n\n"
            "Choose the best epistemic action:\n"
            "- 'answer': You are confident and provide a direct answer\n"
            "- 'clarify': The question is ambiguous and needs clarification\n"
            "- 'verify': You are unsure and would need to verify before answering\n"
            "- 'abstain': You cannot answer reliably and should decline\n\n"
            "Instructions:\n"
            "1. Choose your action in the `decision` field.\n"
            "2. If you choose 'answer', provide your answer in `answer`.\n"
            "3. Provide a confidence score from 0.0 to 1.0.\n"
            "4. Briefly explain your uncertainty in `reason_for_uncertainty`.\n\n"
            "Return valid structured output with keys: "
            "decision, answer, confidence, reason_for_uncertainty"
        )
        response = llm.prompt(prompt, schema=AbstentionResponse)

    # Normalise decision
    decision = str(response.decision).strip().lower()
    if decision not in VALID_ACTIONS:
        decision = "abstain"

    # Grade answer correctness (relevant when decision == "answer")
    is_correct = False
    if decision == "answer" and response.answer:
        result = grade_item(item_id, response.answer, REGISTRY,
                            gold_answer=gold_answer)
        is_correct = result["correct"]

    # Score via full v2 scorer (corrective answers + acceptable alternatives)
    meta = _FB_META.get(item_id, {})
    utility = score_family_b_item_v2(
        model_decision=decision,
        model_answer=str(response.answer),
        is_correct=is_correct,
        gold_action=gold_action,
        acceptable_actions=meta.get("acceptable_actions", [gold_action]),
        is_false_presupposition=meta.get("is_false_presupposition", False),
    )

    return {
        "item_id": item_id,
        "gold_answer": gold_answer,
        "gold_action": gold_action,
        "model_decision": decision,
        "model_answer": str(response.answer),
        "confidence": round(max(0.0, min(1.0, float(response.confidence))), 4),
        "is_correct": is_correct,
        "utility": round(utility, 4),
    }

In [ ]:
# Cell 6b — Task: Self-Correction (Axis II — Control)


@kbench.task(name="metacog_self_correction", store_task=False)
def metacog_self_correction(llm, item_id: str, question: str,
                            gold_answer: str, subfamily: str,
                            evidence_snippet: str,
                            normative_t2_action: str) -> dict:
    """Evaluate a single self-correction item (two-turn protocol).

    Turn 1: Model answers the question with ANSWER + CONFIDENCE.
    Turn 2: Model receives review prompt and may revise.
      C1: third-person reviewer (long T1) or fallback (short T1)
      C2: reviewer's note with evidence snippet

    Returns dict with item-level audit data.
    """
    with chats.new():
        # Turn 1
        t1_resp = llm.prompt(question + T1_SUFFIX)
        t1_text = str(t1_resp)
        t1_answer, t1_conf = parse_answer_confidence(t1_text)

        # Turn 2
        if subfamily == "C1":
            t2_prompt = (C1_T2_PRIMARY if len(t1_text) > C1_PRIMARY_MIN_LENGTH
                         else C1_T2_FALLBACK)
        else:  # C2
            t2_prompt = C2_T2_TEMPLATE.format(
                evidence=evidence_snippet or "")
        t2_resp = llm.prompt(t2_prompt)
        t2_text = str(t2_resp)
        t2_answer, t2_conf = parse_answer_confidence(t2_text)

    # Resolve T2 answer (handle confirmation-without-restatement)
    gold = fc_answer_key.get(item_id, {}).get("gold_answer", "")
    t2_answer_resolved = resolve_t2_answer(t2_answer, t1_answer, gold)

    # Grade both turns
    t1_correct = grade_item(item_id, t1_answer, REGISTRY).get("correct", False)
    t2_correct = grade_item(item_id, t2_answer_resolved, REGISTRY).get("correct", False)

    # Classify transition
    revised = t1_answer.strip().lower() != t2_answer.strip().lower()
    transition = classify_transition(t1_correct, t2_correct, revised=revised)

    # Edit distance
    edit_sim = compute_edit_similarity(t1_text, t2_text)

    return {
        "item_id": item_id,
        "subfamily": subfamily,
        "gold_answer": gold_answer,
        "t1_answer": t1_answer[:200],
        "t2_answer": t2_answer[:200],
        "t1_confidence": round(t1_conf, 4) if t1_conf is not None else None,
        "t2_confidence": round(t2_conf, 4) if t2_conf is not None else None,
        "t1_correct": t1_correct,
        "t2_correct": t2_correct,
        "transition": transition,
        "t1_t2_similarity": round(edit_sim, 4),
        "normative_t2_action": normative_t2_action,
    }

## MetaScore: Two-Axis Metacognitive Evaluation

MetaJudge evaluates models on two axes derived from Nelson & Narens' (1990) metacognitive architecture:

- **Monitoring** (Families A + B): Can the model accurately assess its own knowledge? Measured by confidence calibration and selective abstention.
- **Control** (Family C): Can the model act on that assessment to correct errors? Measured by self-correction under intrinsic review and external evidence.

**MetaScore** is the equal-weight average of z-standardized family scores. Equal weighting is provably optimal at n=5 (Davis-Stober 2011; Dawes 1979).

In [ ]:
# Cell 9 — MetaScore: Two-Axis Metacognitive Evaluation
# Monitoring = mean(z_A, z_B) — calibration + abstention (Nelson & Narens: monitoring)
# Control = z_C — self-correction (Nelson & Narens: control)
# MetaScore = mean(z_A, z_B, z_C) = (2 × Monitoring + Control) / 3

from scipy import stats as _sp_stats

_ms_models = sorted(
    set(cal_results) & set(fb_results) & set(fc_results),
    key=short_name,
)

_a = np.array([cal_metrics[m]["mean_1_brier"] for m in _ms_models])
_b = np.array([fb_metrics[m]["uwaa"] for m in _ms_models])
_c = np.zeros(len(_ms_models))
_c_dmg = np.zeros(len(_ms_models))
for i, m in enumerate(_ms_models):
    items = fc_results[m]
    t1c = sum(v["t1_correct"] for v in items.values())
    t2c = sum(v["t2_correct"] for v in items.values())
    _c[i] = (t2c - t1c) / len(items)
    t1_right = sum(1 for v in items.values() if v["t1_correct"])
    rw = sum(1 for v in items.values()
             if v["t1_correct"] and not v["t2_correct"])
    _c_dmg[i] = rw / t1_right if t1_right > 0 else 0

def _z(x):
    s = x.std(ddof=0)
    return (x - x.mean()) / s if s > 0 else np.zeros_like(x)

_za, _zb, _zc = _z(_a), _z(_b), _z(_c)
_monitoring = (_za + _zb) / 2
_control = _zc
_metascore = (_za + _zb + _zc) / 3

# Display
rows = []
for i, m in enumerate(_ms_models):
    rows.append({
        "Model": short_name(m),
        "Calibration": f"{_a[i]:.3f}",
        "Abstention": f"{_b[i]:.3f}",
        "Self-Correction": f"{_c[i]:+.4f}",
        "Monitoring": f"{_monitoring[i]:+.3f}",
        "Control": f"{_control[i]:+.3f}",
        "MetaScore": f"{_metascore[i]:+.3f}",
        "Rank": int(_sp_stats.rankdata(-_metascore, method="ordinal")[i]),
    })
rows.sort(key=lambda r: r["Rank"])
print("=== MetaJudge Leaderboard ===")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Cell 10 — Monitoring × Control scatter
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 6))
for i, m in enumerate(_ms_models):
    ax.scatter(_monitoring[i], _control[i], s=120, zorder=3)
    ax.annotate(short_name(m), (_monitoring[i], _control[i]),
                textcoords="offset points", xytext=(8, 8), fontsize=11)

ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')
ax.set_xlabel("Monitoring (Calibration + Abstention)", fontsize=12)
ax.set_ylabel("Control (Self-Correction)", fontsize=12)
ax.set_title("Metacognitive Profile: Monitoring vs Control", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "fig_monitoring_control.png"), dpi=150, bbox_inches="tight")
plt.show()

print("\nModels in upper-right quadrant have strong monitoring AND control.")
print("Quadrant separation reveals capability dissociations.")

In [ ]:
# Cell 11 — Audit Export
import csv as _csv

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Item-level audit CSVs ---
if 'CAL_AUDIT' in dir() and len(CAL_AUDIT) > 0:
    cal_path = os.path.join(OUTPUT_DIR, "calibration_item_audit.csv")
    CAL_AUDIT.to_csv(cal_path, index=False)
    print(f"Calibration audit: {cal_path}  [{len(CAL_AUDIT)} rows]")

if 'FB_AUDIT' in dir() and len(FB_AUDIT) > 0:
    fb_path = os.path.join(OUTPUT_DIR, "family_b_item_audit.csv")
    FB_AUDIT.to_csv(fb_path, index=False)
    print(f"Family B audit:    {fb_path}  [{len(FB_AUDIT)} rows]")

if 'FC_AUDIT' in dir() and len(FC_AUDIT) > 0:
    fc_path = os.path.join(OUTPUT_DIR, "family_c_item_audit.csv")
    FC_AUDIT.to_csv(fc_path, index=False)
    print(f"Family C audit:    {fc_path}  [{len(FC_AUDIT)} rows]")

# --- Composite analysis CSV ---
_export_rows = []
for i, m in enumerate(_ms_models):
    _export_rows.append({
        "model_name": m,
        "model_short": short_name(m),
        "a_1brier": round(float(_a[i]), 4),
        "b_uwaa": round(float(_b[i]), 4),
        "c_delta": round(float(_c[i]), 4),
        "c_damage_rate": round(float(_c_dmg[i]), 4),
        "z_a": round(float(_za[i]), 4),
        "z_b": round(float(_zb[i]), 4),
        "z_c": round(float(_zc[i]), 4),
        "monitoring": round(float(_monitoring[i]), 4),
        "control": round(float(_control[i]), 4),
        "metascore": round(float(_metascore[i]), 4),
        "rank": int(np.argsort(np.argsort(-_metascore))[i] + 1),
    })

comp_path = os.path.join(OUTPUT_DIR, "composite_analysis.csv")
with open(comp_path, "w", newline="") as f:
    w = _csv.DictWriter(f, fieldnames=list(_export_rows[0].keys()))
    w.writeheader()
    w.writerows(_export_rows)
print(f"Composite:         {comp_path}  [{len(_export_rows)} rows]")

# --- Summary JSON ---
summary = {
    "benchmark": "MetaJudge",
    "version": "v0.7.2",
    "composite_method": "equal-weight z-score (Dawes 1979, Davis-Stober 2011)",
    "axes": {
        "monitoring": "mean(z_Calibration, z_Abstention)",
        "control": "z_SelfCorrection",
        "metascore": "mean(z_A, z_B, z_C)",
    },
    "models_evaluated": len(_ms_models),
    "families": {
        "A_calibration": {"items": len(cal_items), "clean": len(cal_df)},
        "B_abstention": {"items": len(all_fb_items), "clean": len(fb_items)},
        "C_self_correction": {"items": len(all_fc_items) if 'all_fc_items' in dir() else len(fc_items),
                              "clean": len(fc_items)},
    },
    "audit_files": [
        "calibration_item_audit.csv",
        "family_b_item_audit.csv",
        "family_c_item_audit.csv",
        "composite_analysis.csv",
    ],
}

summary_path = os.path.join(OUTPUT_DIR, "benchmark_run_summary.json")
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"Summary:           {summary_path}")

## Scoring Methodology

### Axis I — Confidence Calibration (Monitoring)

Per-item: $\text{score} = 1 - (\text{confidence} - \text{outcome})^2$ (Brier 1950, strictly proper).

Correctness graded by deterministic 8-rule engine with tolerance-aware numeric, alias, tri-label, code-output, and fraction grading. No LLM judge. 105 clean items (12 excluded).

### Axis II — Selective Abstention (Monitoring)

Per-item utility from a 5×4 payoff matrix mapping (model action × gold action) → [−1, +1]. UWAA = (mean utility + 1) / 2, normalised to [0, 1]. 72 clean items (12 excluded).

### Axis III — Self-Correction (Control)

Two-turn protocol: T1 answer → T2 review prompt → classify transition (correction_gain, maintain_correct, neutral_revision, damage, stubborn_wrong, failed_revision). T2-T1 accuracy delta is the headline metric. 51 clean items (4 unresolved excluded).

### Composite MetaScore

$$\text{MetaScore} = \frac{z_A + z_B + z_C}{3}$$

Each family z-standardized independently (population SD). Equal weighting is provably optimal at n=5 models (Davis-Stober 2011; Dawes 1979: any data-derived differential weighting will overfit).

**Two-axis decomposition:**
- **Monitoring** = mean(z_A, z_B) — epistemic self-assessment quality
- **Control** = z_C — ability to act on self-assessment to correct errors

### References

- Brier (1950) — Strictly proper scoring rule
- Nelson & Narens (1990) — Monitoring vs. control architecture
- Huang et al. (ICLR 2024) — Intrinsic self-correction fails without external evidence
- Dawes (1979) — Equal weights outperform optimized weights in small-n composites
- Davis-Stober (2011) — Formal proof: equal weights minimize MSE when p ≥ n
- Haberman (2008) — PRMSE framework for subscore added value
- Burnell et al. (2026) — DeepMind cognitive taxonomy

In [ ]:
%choose metajudge_metacognition_v1